# GTE embeddings benchmark — French vs. English (STS)

Benchmarks the Databricks Foundation Model API endpoint **`databricks-gte-large-en`**
on the **STS Benchmark**, run in parallel on **English** and **French**, to measure how
much embedding quality the English-tuned GTE model loses on French.

- **Dataset:** `stsb_multi_mt` (`en` and `fr` configs) — the same 1,379 STS-B test pairs
  with identical gold similarity scores (0–5). This is MTEB's French STS task.
- **Metric:** Spearman correlation between cosine similarity and gold score
  (`cosine_spearman`, MTEB's standard STS metric), reported per language + FR/EN ratio.
- **Compute:** serverless. **Model access:** FMAPI only (no local model).
- **Data caching:** downloaded once from HuggingFace into a UC Volume; subsequent runs
  read the cached parquet and skip the download.
- **Charts:** the final cell regenerates the README comparison charts from `results` and
  writes them to the UC Volume, so they stay reproducible from the run.

See `SPEC/SPECS.md` for the full specification.

In [ ]:
# `mlflow.deployments` is the FMAPI client and is not preinstalled on serverless;
# matplotlib is used for the chart cell. Do NOT pass -U: upgrading numpy breaks
# serverless's prebuilt pandas/pyspark.
%pip install -q mlflow-skinny matplotlib
dbutils.library.restartPython()

In [ ]:
# --- Config ---
ENDPOINT = "databricks-gte-large-en"       # Databricks FMAPI embedding endpoint
HF_DATASET = "PhilipMay/stsb_multi_mt"     # parallel multilingual STS-B (namespaced repo id)
LANGS = ["en", "fr"]
SPLIT = "test"                             # 1,379 pairs per language
GOLD_MAX = 5.0                             # STS-B score range is 0..5
BATCH_SIZE = 100                           # texts per FMAPI request

# UC Volume used as the dataset cache + results/chart output. Edit catalog/schema to taste.
CATALOG = "lucasbruand_catalog"
SCHEMA = "gte_french_bench"
VOLUME = "data"
VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
VOLUME_DIR = f"{VOLUME_ROOT}/stsb"
RESULTS_CSV = f"{VOLUME_ROOT}/results_gte_french.csv"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
import os
os.makedirs(VOLUME_DIR, exist_ok=True)
print(f"Cache dir: {VOLUME_DIR}")

In [ ]:
# --- Load STS-B per language, caching to the UC Volume ---
# Download the parquet directly from HuggingFace's parquet API (no `datasets` dependency,
# avoids the serverless /root/.cache permission issue). Download only on cache miss.
import os
import io
import urllib.request
import pandas as pd

def load_sts(lang):
    cache_path = os.path.join(VOLUME_DIR, f"stsb_{lang}_{SPLIT}.parquet")
    if os.path.exists(cache_path):
        print(f"[{lang}] cache hit  -> {cache_path}")
        return pd.read_parquet(cache_path)
    url = f"https://huggingface.co/api/datasets/{HF_DATASET}/parquet/{lang}/{SPLIT}/0.parquet"
    print(f"[{lang}] cache miss -> downloading {url}")
    raw = urllib.request.urlopen(url, timeout=60).read()
    df = pd.read_parquet(io.BytesIO(raw))[["sentence1", "sentence2", "similarity_score"]]
    df.to_parquet(cache_path, index=False)
    print(f"[{lang}] cached {len(df)} rows -> {cache_path}")
    return df

data = {lang: load_sts(lang) for lang in LANGS}
for lang, df in data.items():
    print(f"[{lang}] {df.shape[0]} pairs, score range {df.similarity_score.min()}..{df.similarity_score.max()}")

In [ ]:
# --- FMAPI embedding helper (MLflow deployments client) ---
import time
import numpy as np
from mlflow.deployments import get_deploy_client

_client = get_deploy_client("databricks")

def embed(texts, endpoint=ENDPOINT, batch_size=BATCH_SIZE, max_retries=5):
    """Return an (n, dim) float32 array of L2-normalized embeddings."""
    out = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        for attempt in range(max_retries):
            try:
                resp = _client.predict(endpoint=endpoint, inputs={"input": batch})
                out.extend(item["embedding"] for item in resp["data"])
                break
            except Exception as e:  # rate limit / transient — backoff and retry
                if attempt == max_retries - 1:
                    raise
                wait = 2 ** attempt
                print(f"  batch {start} attempt {attempt+1} failed ({e}); retry in {wait}s")
                time.sleep(wait)
    arr = np.asarray(out, dtype=np.float32)
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return arr / norms

# Smoke test
_probe = embed(["hello world", "bonjour le monde"])
print(f"Embedding dim: {_probe.shape[1]}")

In [ ]:
# --- Run the STS benchmark for each language ---
from scipy.stats import spearmanr, pearsonr

rows = []
for lang in LANGS:
    df = data[lang]
    s1 = df["sentence1"].tolist()
    s2 = df["sentence2"].tolist()
    gold = df["similarity_score"].to_numpy(dtype=np.float32) / GOLD_MAX
    print(f"[{lang}] {len(s1)} pairs — embedding...")

    e1 = embed(s1)
    e2 = embed(s2)
    cos = np.sum(e1 * e2, axis=1)  # both sides already L2-normalized

    spearman = spearmanr(cos, gold).correlation
    pearson = pearsonr(cos, gold)[0]
    rows.append({"lang": lang, "n_pairs": len(s1),
                 "cosine_spearman": spearman, "cosine_pearson": pearson})
    print(f"[{lang}] cosine_spearman={spearman:.4f}  cosine_pearson={pearson:.4f}")

results = pd.DataFrame(rows).set_index("lang")

In [ ]:
# --- Headline: FR/EN ratio + results table ---
ratio = results.loc["fr", "cosine_spearman"] / results.loc["en", "cosine_spearman"]

print("=" * 56)
print(f"Endpoint: {ENDPOINT}")
print(f"Dataset:  {HF_DATASET} [{SPLIT}]  (cached in {VOLUME_DIR})")
print("=" * 56)
print(results.round(4).to_string())
print("-" * 56)
print(f"FR/EN cosine_spearman ratio: {ratio:.3f}")
print("  ~1.0  -> French served about as well as English")
print("  <<1.0 -> English GTE endpoint is a poor fit for French")

results.to_csv(RESULTS_CSV)
print(f"\nSaved: {RESULTS_CSV}")
display(results.reset_index())

In [ ]:
# --- Comparison charts (reproducible from `results`) ---
# Regenerates the two README charts and writes them to the UC Volume. To refresh the
# committed images, copy them from VOLUME_ROOT into the repo's gte-french/assets/.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e5e4e0"
EN_C, FR_C = "#2a78d6", "#eb6834"   # validated categorical pair (blue / orange)
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "font.size": 11,
    "text.color": INK, "axes.edgecolor": GRID, "axes.labelcolor": INK_2,
    "xtick.color": INK_2, "ytick.color": INK_2, "font.family": "DejaVu Sans",
})

metrics = ["Spearman", "Pearson"]
en = [results.loc["en", "cosine_spearman"], results.loc["en", "cosine_pearson"]]
fr = [results.loc["fr", "cosine_spearman"], results.loc["fr", "cosine_pearson"]]
ret = [fr[i] / en[i] for i in range(len(metrics))]
x = np.arange(len(metrics)); w = 0.34

# Chart 1: grouped bars, English vs. French
fig, ax = plt.subplots(figsize=(7.2, 4.2), dpi=150)
for offset, vals, color, label in [(-w/2 - 0.01, en, EN_C, "English"),
                                    (w/2 + 0.01, fr, FR_C, "French")]:
    bars = ax.bar(x + offset, vals, w, label=label, color=color)
    for r in bars:
        ax.text(r.get_x() + r.get_width()/2, r.get_height() + 0.012,
                f"{r.get_height():.3f}", ha="center", va="bottom", fontsize=10, color=INK)
ax.set_ylim(0, 1.0); ax.set_ylabel("correlation (cosine vs. gold)")
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_title(f"GTE ({ENDPOINT}) on STS-B: English vs. French",
             fontsize=12.5, color=INK, pad=12, loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, color=GRID, linewidth=1); ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper right", fontsize=10)
fig.tight_layout(); fig.savefig(f"{VOLUME_ROOT}/sts_en_vs_fr.png", facecolor=SURFACE)

# Chart 2: French retention (fraction of English)
fig2, ax2 = plt.subplots(figsize=(7.2, 2.6), dpi=150)
y = np.arange(len(metrics))
bars = ax2.barh(y, ret, 0.5, color=FR_C)
ax2.axvline(1.0, color=INK_2, linewidth=1.2, linestyle=(0, (4, 3)))
ax2.text(1.0, 1.62, "English = 1.00", color=INK_2, fontsize=9.5, ha="center")
for r, v in zip(bars, ret):
    ax2.text(v - 0.02, r.get_y() + r.get_height()/2, f"{v:.1%}", ha="right", va="center",
             color="white", fontsize=10.5, fontweight="bold")
ax2.set_xlim(0, 1.08); ax2.set_yticks(y); ax2.set_yticklabels(metrics); ax2.invert_yaxis()
ax2.set_xlabel("French score as a fraction of English")
ax2.set_title(f"French retains ~{ret[0]:.0%} of English quality",
              fontsize=12.5, color=INK, pad=10, loc="left")
ax2.spines[["top", "right", "left"]].set_visible(False)
ax2.xaxis.grid(True, color=GRID, linewidth=1); ax2.set_axisbelow(True)
fig2.tight_layout(); fig2.savefig(f"{VOLUME_ROOT}/fr_retention.png", facecolor=SURFACE)

print(f"Charts written to {VOLUME_ROOT}/")
display(fig); display(fig2)